# Modelling

Second notebook, continues from `exploration.ipynb`. That one was about getting to know the data,
this one is about the models: define the methods once, define the knobs we want to turn on them
(scaling, regularization, feature subsets, kernels, ...), push everything through one and the same
evaluation, and then pick one method with a rule we wrote down *before* looking at the numbers.

The notebook is in two halves:

1. **Definitions** (up top): data, preprocessing, the method registry, the variant registry, the evaluation
   protocol, the comparison helpers and some diagnostics plots. Nothing in this half runs a model.
2. **Running things** (down below): select which methods / variants to run, run them, compare, decide,
   refit the winner and write the submission string.

So adding a new model or a new variant of one is one line in a dict, and it automatically gets the same
cross validation, the same tables and the same plots as everything else.

In [ ]:
# Import modules
import numpy as np
import pandas as pd
import random
import json
from dataclasses import dataclass, field, replace

# We use seaborn for prettier plots
import matplotlib.pyplot as plt
import seaborn as sns

# To get typehints
from matplotlib.figure import Figure
from matplotlib.axes import Axes
from numpy.typing import NDArray
from typing import * # bad practice but I dont care

# sklearn: the preprocessing and model selection machinery
from sklearn.base import clone
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (StandardScaler, RobustScaler, QuantileTransformer,
                                   OneHotEncoder, FunctionTransformer, PolynomialFeatures)
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold, GridSearchCV,
                                     cross_validate, cross_val_predict, validation_curve, learning_curve)
from sklearn.inspection import permutation_importance
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay
from scipy import stats

# sklearn: the methods from the lab instructions
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
sns.set_theme()

PALETTE = {0: "red", 1: "blue"}
pd.set_option("display.width", 160, "display.max_columns", 30)

## Data

In [ ]:
# Load data, same as in the exploration
train = pd.read_csv("labs/lab_1_music_taste_prediction/data/training_data.csv")
test = pd.read_csv("labs/lab_1_music_taste_prediction/data/songs_to_classify.csv")

LABEL = "label"
DISCRETE_COLS = ["key", "mode", "time_signature"]
NUMERIC_COLS = [col for col in train.columns if col not in DISCRETE_COLS + [LABEL]]

# From the exploration: a handful of songs appear more than once (identical rows, same label).
# If a duplicated song ends up in both the training and the validation part of a fold, the
# validation score is inflated (kNN and deep trees basically memorize it). So drop them before
# any split happens.
n_before = len(train)
train = train.drop_duplicates().reset_index(drop=True)
print(f"dropped {n_before - len(train)} duplicate rows, {len(train)} songs left, {train[LABEL].mean():.3f} liked")

train_features = train.drop(LABEL, axis=1)
train_labels = train[LABEL]
test_features = test[train_features.columns]   # same column order as train
print(train_features.shape, test_features.shape)

## The protocol, written before any number

Decided here, up front, so that the comparison at the end is a rule being applied and not me
picking the biggest number:

- **Metric:** accuracy, because that is what the leaderboard scores. Balanced accuracy and ROC AUC are
  reported next to it (the classes are 60/40 so a model can look ok by leaning on "like"), but nothing is
  decided on them.
- **Resampling:** repeated stratified k-fold, 5 folds x 5 repeats = 25 outer splits, one fixed seed. Every
  method is scored on exactly the same 25 splits, so comparisons between methods are *paired*.
- **Tuning:** anything with hyperparameters is tuned with a grid search *inside* each outer training part
  (5 inner folds). The outer validation part never sees the tuning. The chosen hyperparameters are
  recorded per split so we can see if the search actually settles.
- **Preprocessing lives inside the pipeline** so scaling and encoding are fit on the training part only.
- **Uncertainty:** the 25 scores are not independent (the splits share most of their songs), so the standard
  error gets the Nadeau-Bengio correction `std * sqrt(1/k + n_val/n_train)` instead of `std/sqrt(k)`.
- **The test file** (`songs_to_classify.csv`) is only touched once, at the very end, by the chosen model.
- **Decision rule:** among the methods whose mean accuracy is within one corrected standard error of the
  best method (error of the *paired* per-split difference), take the one with the fewest tuned
  hyperparameters. Ties broken by a simplicity order fixed below.
- **Screening vs full:** the wide grid of variants is first run with 2 repeats (the "screen", 10 splits).
  A variant only gets the full 5 repeats if it beats its base method on the screen by more than the error
  of that gap ("promotion"). Screen numbers are never mixed with full-protocol numbers in a table.

In [ ]:
N_SPLITS = 5
N_REPEATS = 5           # the full protocol
SCREEN_REPEATS = 2      # the cheaper screen for the wide variant grid
N_INNER = 5
PRIMARY = "accuracy"
SCORES = ["accuracy", "balanced_accuracy", "roc_auc"]

# simplest first, used to break ties in the decision rule
SIMPLICITY = ["logreg", "lda", "qda", "knn", "tree", "svm_linear", "svm_rbf", "bagging", "rf", "boosting", "adaboost"]

def outer_cv(n_repeats: int = N_REPEATS) -> RepeatedStratifiedKFold:
    return RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=n_repeats, random_state=SEED)

def inner_cv() -> StratifiedKFold:
    return StratifiedKFold(n_splits=N_INNER, shuffle=True, random_state=SEED)

def corrected_se(scores: pd.Series) -> float:
    """
    Nadeau & Bengio corrected standard error of a mean over k cross validation scores.
    n_val / n_train = 1 / (N_SPLITS - 1) for k-fold. Works on paired differences too.
    """
    k = len(scores)
    return float(np.sqrt((1 / k + 1 / (N_SPLITS - 1)) * scores.var(ddof=1)))

## Preprocessing

One function builds the `ColumnTransformer` every pipeline starts with. The arguments are the knobs
the variants turn: which scaler, which numeric columns get a `log1p` first (the spiky ones from the
exploration), how the discrete columns are encoded, an optional feature subset, and optional pairwise
interaction terms between the numeric features.

In [ ]:
# from the exploration
SPIKY_COLS = ["duration", "speechiness", "instrumentalness", "liveness"]   # spike + long tail
TOP_COLS = ["speechiness", "loudness", "acousticness", "energy"]           # the four strong ones

def make_scaler(kind: str):
    """standard | robust | quantile | none"""
    if kind == "standard":
        return StandardScaler()
    if kind == "robust":
        return RobustScaler()
    if kind == "quantile":
        # rank transform to a normal shape, kills the tails completely
        return QuantileTransformer(output_distribution="normal", n_quantiles=200, random_state=SEED)
    if kind == "none":
        return "passthrough"
    raise ValueError(kind)

def make_encoder(kind: str):
    """onehot | grouped (rare levels share one column) | drop (no discrete columns at all)"""
    if kind == "onehot":
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    if kind == "grouped":
        return OneHotEncoder(handle_unknown="infrequent_if_exist", sparse_output=False, min_frequency=0.03)
    if kind == "drop":
        return "drop"
    raise ValueError(kind)

def make_preprocessor(
    scaler: str = "standard",
    log_cols: Sequence[str] = (),
    encode: str = "onehot",
    features: Optional[Sequence[str]] = None,
    interactions: bool = False,
) -> ColumnTransformer:
    """
    numeric columns -> (log1p ->) scaler (-> pairwise products)
    discrete columns -> one-hot
    everything else -> dropped
    """
    features = list(train_features.columns) if features is None else list(features)
    numeric = [c for c in NUMERIC_COLS if c in features]
    discrete = [c for c in DISCRETE_COLS if c in features]

    num_tf = make_scaler(scaler)
    if interactions:
        num_tf = make_pipeline(
            StandardScaler() if num_tf == "passthrough" else num_tf,
            PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
            StandardScaler(),
        )

    plain = [c for c in numeric if c not in log_cols]
    logged = [c for c in numeric if c in log_cols]
    branches = [("num", num_tf, plain)]
    if logged:
        branches.append(("log", make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), clone(num_tf)), logged))
    branches.append(("cat", make_encoder(encode), discrete))
    return ColumnTransformer(branches, remainder="drop", verbose_feature_names_out=False)

# quick check: shapes
for kwargs in [dict(), dict(encode="drop"), dict(features=TOP_COLS), dict(interactions=True)]:
    print(kwargs, "->", make_preprocessor(**kwargs).fit_transform(train_features).shape[1], "columns")

## Method registry

Every method from the lab instructions, as *what differs between them*: the estimator and the grid the
protocol tunes. The grids are small on purpose (a handful of values per knob), we have 736 songs and
don't want to spend an hour on a grid that a flat validation curve would not reward anyway.
Grid keys are pipeline parameter names (`clf__C`) so they go straight into `GridSearchCV`.

In [ ]:
@dataclass
class Method:
    make: Callable[[], Any]                                 # returns an unfitted estimator
    grid: Dict[str, List] = field(default_factory=dict)     # what the inner search tunes

METHODS: Dict[str, Method] = {
    # the floor: always say "like"
    "dummy": Method(lambda: DummyClassifier(strategy="most_frequent")),
    # L2 by default; C is the inverse regularization strength, log grid from heavy to almost none
    "logreg": Method(lambda: LogisticRegression(max_iter=5000, random_state=SEED),
                     {"clf__C": list(np.logspace(-3, 3, 13))}),
    "knn": Method(lambda: KNeighborsClassifier(),
                  {"clf__n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21, 31, 41, 51], "clf__weights": ["uniform", "distance"]}),
    # one shared covariance -> linear boundary, nothing to tune
    "lda": Method(lambda: LinearDiscriminantAnalysis()),
    # one covariance per class over ~28 columns (one-hot included) is singular without shrinkage
    "qda": Method(lambda: QuadraticDiscriminantAnalysis(reg_param=0.1),
                  {"clf__reg_param": [0.01, 0.05, 0.1, 0.25, 0.5, 0.9]}),
    "tree": Method(lambda: DecisionTreeClassifier(random_state=SEED),
                   {"clf__max_depth": [2, 3, 4, 6, 8, None], "clf__min_samples_leaf": [1, 5, 10, 20]}),
    # enough trees that more only changes noise, tune the two knobs that set each tree's variance
    "rf": Method(lambda: RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED),
                 {"clf__max_features": ["sqrt", 0.25, 0.5], "clf__min_samples_leaf": [1, 3, 10]}),
    # a forest that sees every feature at every split, so rf - bagging = what decorrelating buys
    "bagging": Method(lambda: BaggingClassifier(DecisionTreeClassifier(), n_estimators=300, n_jobs=-1, random_state=SEED),
                      {"clf__estimator__min_samples_leaf": [1, 3, 10]}),
    # sklearn's histogram gradient boosting, the same additive model of shallow trees as classic GB
    "boosting": Method(lambda: HistGradientBoostingClassifier(random_state=SEED),
                       {"clf__learning_rate": [0.03, 0.1, 0.3], "clf__max_depth": [2, 3, None], "clf__max_iter": [100, 300]}),
    # the textbook one: stumps, reweighting the songs the last ones got wrong
    "adaboost": Method(lambda: AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), random_state=SEED),
                       {"clf__learning_rate": [0.1, 0.3, 1.0], "clf__n_estimators": [100, 300]}),
    "svm_linear": Method(lambda: SVC(kernel="linear"), {"clf__C": [0.001, 0.01, 0.1, 1.0, 10.0]}),
    "svm_rbf": Method(lambda: SVC(kernel="rbf"), {"clf__C": [0.1, 1.0, 10.0, 100.0], "clf__gamma": [0.01, 0.03, 0.1, 0.3]}),
}

## Variant registry

A variant = a method + what we change about how it is run. The plain variants (same name as the method)
are the baselines every other variant of that method is compared against. Every variant builds the same
shape of pipeline, `preprocessor -> clf`, so the evaluation below does not care which one it gets.

In [ ]:
@dataclass
class Variant:
    method: str
    scaler: str = "standard"
    log_cols: Tuple[str, ...] = ()
    encode: str = "onehot"
    features: Optional[Tuple[str, ...]] = None     # None = all features
    interactions: bool = False
    params: Dict[str, Any] = field(default_factory=dict)   # set on the estimator (e.g. a different kernel)
    grid: Optional[Dict[str, List]] = None                 # replaces the method's grid when given
    note: str = ""

    def pipeline(self) -> Pipeline:
        clf = METHODS[self.method].make()
        if self.params:
            clf.set_params(**self.params)
        pre = make_preprocessor(self.scaler, self.log_cols, self.encode, self.features, self.interactions)
        return Pipeline([("pre", pre), ("clf", clf)])

    def search_grid(self) -> Dict[str, List]:
        return METHODS[self.method].grid if self.grid is None else self.grid

    def n_tuned(self) -> int:
        return len(self.search_grid())


VARIANTS: Dict[str, Variant] = {}

def add_variants(**variants: Variant) -> None:
    for name, v in variants.items():
        assert name not in VARIANTS, f"duplicate variant {name}"
        VARIANTS[name] = v

# the plain ones = the method as is
add_variants(**{name: Variant(name) for name in METHODS})

# linear family: what regularization, scaling and features do to a line
add_variants(
    logreg_l1=Variant("logreg", params=dict(solver="saga", l1_ratio=1.0, max_iter=20000), note="L1: coefficients go to exactly 0 -> feature selection"),
    logreg_elastic=Variant("logreg", params=dict(solver="saga", l1_ratio=0.5, max_iter=20000), note="half L1 half L2"),
    logreg_robust=Variant("logreg", scaler="robust", note="median/IQR scaling, tails pull less"),
    logreg_quantile=Variant("logreg", scaler="quantile", note="rank -> normal, no tails at all"),
    logreg_log=Variant("logreg", log_cols=tuple(SPIKY_COLS), note="log1p on the spiky four"),
    logreg_top4=Variant("logreg", features=tuple(TOP_COLS), note="only the four strong features"),
    logreg_numeric=Variant("logreg", encode="drop", note="what key/mode/meter are worth to a line"),
    logreg_grouped=Variant("logreg", encode="grouped", note="rare key and meter levels grouped"),
    logreg_interactions=Variant("logreg", interactions=True, note="pairwise products, the line can bend along pairs"),
    lda_shrink=Variant("lda", params=dict(solver="lsqr", shrinkage="auto"), note="Ledoit-Wolf shrinkage of the covariance"),
    lda_top4=Variant("lda", features=tuple(TOP_COLS)),
    qda_numeric=Variant("qda", encode="drop", note="no one-hot columns in the class covariances"),
    qda_top4=Variant("qda", features=tuple(TOP_COLS)),
)

# kNN family: a distance is only as good as the space it is measured in
add_variants(
    knn_robust=Variant("knn", scaler="robust"),
    knn_quantile=Variant("knn", scaler="quantile"),
    knn_log=Variant("knn", log_cols=tuple(SPIKY_COLS)),
    knn_numeric=Variant("knn", encode="drop", note="no one-hot axes in the distance"),
    knn_top4=Variant("knn", features=tuple(TOP_COLS)),
    knn_manhattan=Variant("knn", params=dict(p=1), note="L1 distance"),
)

# SVM family: the kernel, the margin, and the scale the kernel sees
add_variants(
    svm_poly=Variant("svm_rbf", params=dict(kernel="poly", coef0=1.0),
                     grid={"clf__C": [0.1, 1.0, 10.0], "clf__degree": [2, 3]}, note="polynomial kernel"),
    svm_rbf_robust=Variant("svm_rbf", scaler="robust"),
    svm_rbf_quantile=Variant("svm_rbf", scaler="quantile"),
    svm_rbf_log=Variant("svm_rbf", log_cols=tuple(SPIKY_COLS)),
    svm_rbf_top4=Variant("svm_rbf", features=tuple(TOP_COLS)),
    svm_rbf_numeric=Variant("svm_rbf", encode="drop"),
)

# tree family: scaling should not matter, features and depth should
add_variants(
    tree_noscale=Variant("tree", scaler="none", note="must give exactly the same splits as tree"),
    tree_top4=Variant("tree", features=tuple(TOP_COLS)),
    rf_top4=Variant("rf", features=tuple(TOP_COLS)),
    rf_numeric=Variant("rf", encode="drop"),
    rf_grouped=Variant("rf", encode="grouped"),
    rf_shallow=Variant("rf", grid={"clf__max_features": ["sqrt", 0.25, 0.5], "clf__max_depth": [3, 5, 8]}, note="depth-limited instead of leaf-limited"),
    boosting_slow=Variant("boosting", grid={"clf__learning_rate": [0.01, 0.03], "clf__max_depth": [2, 3], "clf__max_iter": [300, 600]}, note="smaller steps, more of them"),
    boosting_top4=Variant("boosting", features=tuple(TOP_COLS)),
)

FAMILIES: Dict[str, List[str]] = {
    "linear": [n for n, v in VARIANTS.items() if v.method in ("logreg", "lda", "qda")],
    "knn":    [n for n, v in VARIANTS.items() if v.method == "knn"],
    "svm":    [n for n, v in VARIANTS.items() if v.method.startswith("svm")],
    "trees":  [n for n, v in VARIANTS.items() if v.method in ("tree", "rf", "bagging", "boosting", "adaboost")],
}
print({f: len(v) for f, v in FAMILIES.items()}, "variants in total:", len(VARIANTS))

## Evaluation

One function, `evaluate`, that every variant goes through. `cross_validate` does the outer loop and
`GridSearchCV` the inner one, nothing hand rolled. The result is a tidy frame with one row per outer
split: the scores, the hyperparameters the search picked on that split, the fit time. `run` caches
results per (variant, n_repeats) in a dict so re-running a cell does not recompute, and `save_results` /
`load_results` keep them on disk between sessions.

In [ ]:
RESULTS_PATH = "labs/lab_1_music_taste_prediction/modelling_results.csv"
results: Dict[Tuple[str, int], pd.DataFrame] = {}   # (variant, n_repeats) -> rows

def estimator_for(name: str):
    """the variant's pipeline, wrapped in the inner grid search when it has something to tune"""
    v = VARIANTS[name]
    pipe = v.pipeline()
    if not v.search_grid():
        return pipe
    return GridSearchCV(pipe, v.search_grid(), scoring=PRIMARY, cv=inner_cv(), refit=True)

def chosen_params(fitted) -> str:
    """what the inner search picked, as a stable string we can group by"""
    if not isinstance(fitted, GridSearchCV):
        return "{}"
    return json.dumps({k.replace("clf__", ""): v for k, v in fitted.best_params_.items()}, sort_keys=True, default=str)

def evaluate(name: str, n_repeats: int = N_REPEATS) -> pd.DataFrame:
    """one variant through the protocol -> one row per outer split"""
    cv = cross_validate(estimator_for(name), train_features, train_labels, cv=outer_cv(n_repeats),
                        scoring=SCORES, return_estimator=True, error_score="raise")
    split = np.arange(N_SPLITS * n_repeats)
    return pd.DataFrame({
        "variant": name,
        "method": VARIANTS[name].method,
        "n_repeats": n_repeats,
        "repeat": split // N_SPLITS,
        "fold": split % N_SPLITS,
        **{s: cv[f"test_{s}"] for s in SCORES},
        "params": [chosen_params(e) for e in cv["estimator"]],
        "fit_time": cv["fit_time"],
    })

def run(names: Sequence[str], n_repeats: int = N_REPEATS, verbose: bool = True) -> pd.DataFrame:
    """evaluate what is not cached yet, return the rows of all the names asked for"""
    for name in names:
        if (name, n_repeats) in results:
            continue
        if verbose:
            print(f"{name:22s} ...", end="", flush=True)
        rows = evaluate(name, n_repeats)
        results[(name, n_repeats)] = rows
        if verbose:
            print(f" {rows[PRIMARY].mean():.3f} +- {corrected_se(rows[PRIMARY]):.3f}  ({rows['fit_time'].sum():.0f}s)")
    return pd.concat([results[(n, n_repeats)] for n in names], ignore_index=True)

def save_results(path: str = RESULTS_PATH) -> None:
    pd.concat(results.values(), ignore_index=True).drop(columns="fit_time").to_csv(path, index=False)

def load_results(path: str = RESULTS_PATH) -> None:
    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        return
    for (name, k), rows in df.groupby(["variant", "n_repeats"]):
        if name in VARIANTS:
            results[(name, k)] = rows.assign(fit_time=np.nan).reset_index(drop=True)
    print(f"loaded {len(results)} cached runs")

### Comparing

`summarize` gives mean and corrected error per variant, and the gap to the variant's own base (the plain
method) on the same splits. `paired_vs_best` compares everything to the best variant on the per-split
differences (corrected t-test, like sklearn's model comparison example). `decide` applies the rule from
the protocol section.

In [ ]:
def summarize(rows: pd.DataFrame) -> pd.DataFrame:
    """mean +- corrected se per variant and score, plus the paired gap to its base method"""
    g = rows.groupby("variant", sort=False)
    out = pd.DataFrame({"method": g["method"].first(), "tuned": [VARIANTS[n].n_tuned() for n in g.groups]})
    for s in SCORES:
        out[f"{s} mean"] = g[s].mean()
        out[f"{s} se"] = g[s].apply(corrected_se)
    wide = rows.pivot_table(index=["repeat", "fold"], columns="variant", values=PRIMARY)
    gaps, ses = {}, {}
    for name in wide.columns:
        base = VARIANTS[name].method
        if base in wide.columns and base != name:
            d = wide[name] - wide[base]
            gaps[name], ses[name] = d.mean(), corrected_se(d)
        else:
            gaps[name], ses[name] = 0.0, 0.0
    out["gap to base"] = pd.Series(gaps)
    out["se of gap"] = pd.Series(ses)
    return out.sort_values(f"{PRIMARY} mean", ascending=False)

def paired_vs_best(rows: pd.DataFrame, rope: float = 0.01) -> pd.DataFrame:
    """
    every variant minus the best (non-dummy) one on the same splits: mean gap, corrected se of the gap,
    corrected t-test p-value, and the posterior probabilities that the best is better by more than `rope`,
    that they are practically equal, and that the variant is better.
    """
    wide = rows.pivot_table(index=["repeat", "fold"], columns="variant", values=PRIMARY)
    best = wide.drop(columns="dummy", errors="ignore").mean().idxmax()
    out = []
    for name in wide.columns:
        d = wide[name] - wide[best]
        k, gap, se = len(d), d.mean(), corrected_se(d)
        if name == best or se == 0:
            out.append(dict(variant=name, gap=gap, se=se, p=1.0, p_best_better=0.0, p_equal=1.0, wins=0))
            continue
        post = stats.t(df=k - 1, loc=gap, scale=se)
        out.append(dict(variant=name, gap=gap, se=se,
                        p=2 * stats.t.sf(abs(gap / se), df=k - 1),
                        p_best_better=post.cdf(-rope), p_equal=post.cdf(rope) - post.cdf(-rope),
                        wins=int((d < 0).sum())))
    table = pd.DataFrame(out).set_index("variant")
    table["p_variant_better"] = 1 - table["p_best_better"] - table["p_equal"]
    return table.sort_values("gap", ascending=False).rename_axis(f"vs {best}")

def decide(rows: pd.DataFrame, band: float = 1.0) -> Tuple[str, pd.DataFrame]:
    """
    the rule: among non-dummy variants within `band` corrected errors (of the paired gap) of the best,
    the one with the fewest tuned hyperparameters; ties by SIMPLICITY of the underlying method.
    """
    table = paired_vs_best(rows).drop(index="dummy", errors="ignore")
    ok = table[-table["gap"] <= band * table["se"]].copy()
    ok["tuned"] = [VARIANTS[n].n_tuned() for n in ok.index]
    ok["order"] = [SIMPLICITY.index(VARIANTS[n].method) for n in ok.index]
    ok = ok.sort_values(["tuned", "order"])
    return ok.index[0], ok.drop(columns="order")

def promoted(rows: pd.DataFrame) -> pd.DataFrame:
    """per method: the best variant, kept only if it beats its base by more than the error of the gap"""
    table = summarize(rows)
    best = table.loc[table.groupby("method")[f"{PRIMARY} mean"].idxmax()]
    return best[best["gap to base"] > best["se of gap"]]

def plot_splits(rows: pd.DataFrame, ax: Optional[Axes] = None) -> Axes:
    """one box per variant = its per-split accuracies, best on top"""
    order = rows.groupby("variant")[PRIMARY].mean().sort_values(ascending=False).index
    ax = sns.boxplot(rows, x=PRIMARY, y="variant", order=order, hue="method", dodge=False, showmeans=True, ax=ax)
    ax.figure.set_size_inches(8, 0.35 * len(order) + 1)
    ax.set(xlabel=f"{PRIMARY} on each outer validation fold", ylabel="")
    return ax

### Diagnostics

The plots that say *why* a variant scored as it did. All of them are sklearn routines (`validation_curve`,
`learning_curve`, `permutation_importance`, `cross_val_predict`) called on a variant's pipeline with the
protocol's folds, and one seaborn call to draw.

In [ ]:
def plot_validation_curve(name: str, param: str, grid: Sequence, n_repeats: int = SCREEN_REPEATS, ax: Optional[Axes] = None) -> Axes:
    """one hyperparameter swept with the rest at defaults, train and validation score"""
    tr, va = validation_curve(VARIANTS[name].pipeline(), train_features, train_labels, param_name=param,
                              param_range=list(grid), cv=outer_cv(n_repeats), scoring=PRIMARY, n_jobs=-1)
    long = pd.DataFrame({"value": np.repeat(list(grid), tr.shape[1]),
                         "train": tr.ravel(), "validation": va.ravel()}).melt(id_vars="value", var_name="set", value_name=PRIMARY)
    ax = sns.lineplot(long, x="value", y=PRIMARY, hue="set", errorbar=("se", 1), marker="o", ax=ax)
    values = np.asarray(list(grid), dtype=object)
    if np.issubdtype(np.asarray(list(grid)).dtype, np.number) and min(grid) > 0 and max(grid) / min(grid) > 50:
        ax.set_xscale("log")
    ax.set_title(f"{name}: {param.replace('clf__', '')}")
    return ax

def plot_learning_curve(name: str, n_repeats: int = SCREEN_REPEATS, sizes=(0.2, 0.4, 0.6, 0.8, 1.0), ax: Optional[Axes] = None) -> Axes:
    """does more data help? validation still rising = variance, both lines flat and met = bias"""
    n, tr, va = learning_curve(VARIANTS[name].pipeline(), train_features, train_labels, train_sizes=list(sizes),
                               cv=outer_cv(n_repeats), scoring=PRIMARY, shuffle=True, random_state=SEED, n_jobs=-1)
    long = pd.DataFrame({"n_train": np.repeat(n, tr.shape[1]), "train": tr.ravel(), "validation": va.ravel()}).melt(id_vars="n_train", var_name="set", value_name=PRIMARY)
    ax = sns.lineplot(long, x="n_train", y=PRIMARY, hue="set", errorbar=("se", 1), marker="o", ax=ax)
    ax.set_title(f"{name}: learning curve")
    return ax

def permutation_importances(names: Sequence[str], n_repeats: int = SCREEN_REPEATS, n_shuffles: int = 10, tuned: bool = False) -> pd.DataFrame:
    """
    feature x variant: mean drop in validation accuracy when that original column is shuffled,
    over the outer folds. Rows ordered by mean rank across variants -> the robustly important ones on top.
    """
    rows = []
    for name in names:
        for tr, va in outer_cv(n_repeats).split(train_features, train_labels):
            est = estimator_for(name) if tuned else VARIANTS[name].pipeline()
            est.fit(train_features.iloc[tr], train_labels.iloc[tr])
            pi = permutation_importance(est, train_features.iloc[va], train_labels.iloc[va], scoring=PRIMARY,
                                        n_repeats=n_shuffles, random_state=SEED, n_jobs=-1)
            rows += [dict(variant=name, feature=f, importance=m) for f, m in zip(train_features.columns, pi.importances_mean)]
    wide = pd.DataFrame(rows).groupby(["feature", "variant"])["importance"].mean().unstack("variant")[list(names)]
    return wide.loc[wide.rank(ascending=False).mean(axis=1).sort_values().index]

def oof_scores(name: str, tuned: bool = True) -> pd.DataFrame:
    """out-of-fold score for every song from one stratified k-fold (each song predicted exactly once)"""
    est = estimator_for(name) if tuned else VARIANTS[name].pipeline()
    clf = VARIANTS[name].pipeline().named_steps["clf"]
    how = "predict_proba" if hasattr(clf, "predict_proba") else "decision_function"
    score = cross_val_predict(est, train_features, train_labels, cv=StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED), method=how, n_jobs=-1)
    score = score[:, 1] if score.ndim == 2 else score
    return pd.DataFrame({"variant": name, "y": train_labels.values, "score": score, "is_proba": how == "predict_proba"})

def plot_roc_and_confusion(names: Sequence[str]) -> Figure:
    fig, axs = plt.subplots(1, 1 + len(names), figsize=(5 * (1 + len(names)), 4.5), constrained_layout=True)
    for name in names:
        oof = oof_scores(name)
        RocCurveDisplay.from_predictions(oof["y"], oof["score"], name=name, ax=axs[0])
    axs[0].set_title("out-of-fold ROC")
    for ax, name in zip(axs[1:], names):
        oof = oof_scores(name)
        thr = 0.5 if oof["is_proba"].iat[0] else 0.0
        ConfusionMatrixDisplay.from_predictions(oof["y"], (oof["score"] >= thr).astype(int), display_labels=["dislike", "like"], ax=ax, colorbar=False)
        ax.set_title(f"{name}: out-of-fold confusion")
    return fig

def decision_sensitivity(rows: pd.DataFrame) -> pd.DataFrame:
    """what the rule would pick under its neighbours, so we know how much turns on the strictness"""
    out = {}
    for label, band in [("within 1 se (the rule)", 1.0), ("within 2 se", 2.0), ("within 3 se", 3.0), ("best mean, no band", 0.0)]:
        choice, ok = decide(rows, band)
        out[label] = dict(chosen=choice, qualifying=", ".join(ok.index))
    return pd.DataFrame(out).T

---
# Running things

From here on it is selecting and running. Costs on a 4 core laptop, roughly: the three baselines a minute,
the full sweep of the plain methods ~30-40 minutes (the forests dominate), a family of variants under the
screen 5-20 minutes. `save_results()` after a long cell and `load_results()` at the start of a session.

In [ ]:
load_results()

## Baselines: the floor and the two simplest real methods

In [ ]:
baseline_rows = run(["dummy", "logreg", "knn"])
summarize(baseline_rows).round(3)

## Every method family through the full protocol

The plain variant of every method in the lab instructions, 25 splits each. This is the table the
decision is made on.

In [ ]:
sweep_names = list(METHODS)
sweep_rows = run(sweep_names)
save_results()
summarize(sweep_rows).round(3)

In [ ]:
plot_splits(sweep_rows);

In [ ]:
paired_vs_best(sweep_rows).round(3)

In [ ]:
choice, qualifying = decide(sweep_rows)
print("chosen by the rule:", choice)
qualifying.round(3)

In [ ]:
decision_sensitivity(sweep_rows)

## Variants under the screen, one family at a time

Same splits (the first two repeats of the same seed), so everything is still paired within the screen.
Change `family` and rerun. `promoted` says which variant, if any, earned the full protocol.

In [ ]:
family = "linear"
screen_rows = run(FAMILIES[family], n_repeats=SCREEN_REPEATS)
save_results()
summarize(screen_rows).round(3)

In [ ]:
plot_splits(screen_rows);

In [ ]:
# everything screened so far, and what would be promoted
all_screen = pd.concat([r for (n, k), r in results.items() if k == SCREEN_REPEATS], ignore_index=True)
promoted(all_screen).round(3)

## Promotion: the variants that earned the full protocol

Run them with 5 repeats and let them join the sweep table on equal footing; then the rule decides again
on the union. (Empty if nothing beat its base on the screen, which is a finding in itself.)

In [ ]:
promoted_names = list(promoted(all_screen).index)
print("promoted:", promoted_names)
if promoted_names:
    full_rows = pd.concat([sweep_rows, run(promoted_names)], ignore_index=True)
    save_results()
    display(summarize(full_rows).round(3))
    display(paired_vs_best(full_rows).round(3))
    choice, qualifying = decide(full_rows)
    print("chosen by the rule, on the union:", choice)

## Why: validation curves, learning curves, importances

Pick the ones that matter for the write-up.

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(16, 8), constrained_layout=True)
axs = np.atleast_1d(axs).ravel()
plot_validation_curve("logreg", "clf__C", [1e-3, 1e-2, 1e-1, 1, 10, 100], ax=axs[0])
plot_validation_curve("knn", "clf__n_neighbors", [1, 3, 5, 9, 15, 25, 41, 61], ax=axs[1])
plot_validation_curve("svm_rbf", "clf__gamma", [1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1], ax=axs[2])
plot_validation_curve("tree", "clf__max_depth", [1, 2, 3, 4, 6, 8, 12, 20], ax=axs[3])
plot_validation_curve("rf", "clf__min_samples_leaf", [1, 2, 3, 5, 10, 20], ax=axs[4])
plot_validation_curve("boosting", "clf__learning_rate", [0.01, 0.03, 0.1, 0.3, 1.0], ax=axs[5])

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(16, 8), sharey=True, constrained_layout=True)
for ax, name in zip(np.atleast_1d(axs).ravel(), ["logreg", "knn", "svm_rbf", "tree", "rf", "boosting"]):
    plot_learning_curve(name, ax=ax)

In [ ]:
importances = permutation_importances(["logreg", "lda", "knn", "svm_rbf", "rf", "boosting"])
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(importances, annot=True, fmt=".3f", cmap="Blues", cbar=False, ax=ax)
ax.set_title("permutation importance: drop in held-out accuracy when the feature is shuffled")
importances.round(3)

In [ ]:
plot_roc_and_confusion(["logreg", choice]);

## Production: refit the chosen method and predict the 200 songs

The one and only time the test file is used. The same grid search picks the hyperparameters on all
the training songs, then one prediction per test song in file order, as the string of 0/1 the
leaderboard wants.

In [ ]:
final_model = estimator_for(choice).fit(train_features, train_labels)
if isinstance(final_model, GridSearchCV):
    print(choice, "refit with", chosen_params(final_model), f"(inner cv accuracy {final_model.best_score_:.3f})")

prediction = final_model.predict(test_features)
submission = "".join(str(int(p)) for p in prediction)
assert len(submission) == 200 and set(submission) <= {"0", "1"}
print(f"{submission.count('1')} liked of {len(submission)}")
print(submission)

with open("labs/lab_1_music_taste_prediction/submission.txt", "w") as f:
    f.write(submission + "\n")